## __DESeq2 Gene Selection__

Differential expression analysis (Tumor vs. Control) on TCGA-BRCA RNA-seq count data, used to select a set of significant genes (padj < 0.05, |log2FC| > 2) as an alternative, biologically-motivated feature set for the survival model.

### Imports

In [ ]:
import pandas as pd, os
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference
from dotenv import load_dotenv

load_dotenv("project.env")

True

### Load RNA-seq data

Control and tumor expression matrices, produced by `4-RNA_parquet.ipynb`.

In [2]:
Control = pd.read_parquet(os.path.join(os.getenv("Output_path"),'RNA_seq_expression_Control_data.parquet'), engine='fastparquet')
Tumeric = pd.read_parquet(os.path.join(os.getenv("Output_path"), 'RNA_seq_expression_Tumeric_data.parquet'), engine='fastparquet')

In [4]:
Control.head()

,gene_id,gene_name,unstranded,file_id
0,ENSG00000000003.15,TSPAN6,3116,040d00da-2bc9-49d4-bf6c-b4515b6a2bbf
1,ENSG00000000005.6,TNMD,817,040d00da-2bc9-49d4-bf6c-b4515b6a2bbf
2,ENSG00000000419.13,DPM1,1408,040d00da-2bc9-49d4-bf6c-b4515b6a2bbf
3,ENSG00000000457.14,SCYL3,954,040d00da-2bc9-49d4-bf6c-b4515b6a2bbf
4,ENSG00000000460.17,C1orf112,125,040d00da-2bc9-49d4-bf6c-b4515b6a2bbf


In [5]:
Tumeric.head()

,gene_id,gene_name,unstranded,file_id
0,ENSG00000000003.15,TSPAN6,1635,00b13ccf-ad7c-4613-8366-7c583a399691
1,ENSG00000000005.6,TNMD,101,00b13ccf-ad7c-4613-8366-7c583a399691
2,ENSG00000000419.13,DPM1,1565,00b13ccf-ad7c-4613-8366-7c583a399691
3,ENSG00000000457.14,SCYL3,1183,00b13ccf-ad7c-4613-8366-7c583a399691
4,ENSG00000000460.17,C1orf112,419,00b13ccf-ad7c-4613-8366-7c583a399691


### Build combined count matrix

Combine control and tumor samples into a single count matrix (genes x samples), then drop low-count genes (total count <= 20 across all samples).

In [6]:
combined = pd.concat([
    Control.assign(sample_group = "Control_group"), 
    Tumeric.assign(sample_group = "Tumeric_group")
], ignore_index=True)

count_matrix = combined.pivot_table(
    index="gene_id",
    columns="file_id",
    values="unstranded"
)

count_matrix_pv = count_matrix.T
genes_keep = count_matrix_pv.columns[count_matrix_pv.sum(axis=0) > 20]
count_pv_sel = count_matrix_pv[genes_keep]

### Build sample metadata for DESeq2

One row per sample, labeling each as `Tumeric_group` or `Control_group`, aligned to the count matrix's sample order.

In [9]:
col_data = combined[['file_id','sample_group']].drop_duplicates().set_index('file_id')
col_data = col_data.reindex(count_matrix.columns)

### DESeq2 run and Wald test

Gene expression analysis between Tumor and Control groups.

In [10]:
inference = DefaultInference(n_cpus=4)
dds = DeseqDataSet(
    counts = count_pv_sel, 
    metadata=col_data, 
    design='sample_group', 
    refit_cooks=True, 
    inference=inference
)
dds.deseq2()

Fitting size factors...
... done in 0.17 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 5.18 seconds.

Fitting dispersion trend curve...
... done in 0.19 seconds.

Fitting MAP dispersions...
... done in 5.92 seconds.

Fitting LFCs...
... done in 2.84 seconds.

Calculating cook's distance...
... done in 0.27 seconds.

Replacing 2065 outlier genes.

Fitting dispersions...
... done in 0.56 seconds.

Fitting MAP dispersions...
... done in 0.53 seconds.

Fitting LFCs...
... done in 0.47 seconds.



In [11]:
ds = DeseqStats(dds, contrast=['sample_group', 'Tumeric_group', 'Control_group'])
ds.summary()

Running Wald tests...


Log2 fold change & Wald test p-value: sample_group Tumeric_group vs Control_group
                       baseMean  log2FoldChange     lfcSE       stat  \
gene_id                                                                
ENSG00000000003.15  3325.531567       -0.599214  0.122591  -4.887913   
ENSG00000000005.6    170.141627       -4.336403  0.275804 -15.722780   
ENSG00000000419.13  2383.677820        0.500536  0.077919   6.423820   
ENSG00000000457.14  1596.702607        0.396821  0.075229   5.274846   
ENSG00000000460.17   696.673383        1.294277  0.103595  12.493602   
...                         ...             ...       ...        ...   
ENSG00000288658.1     25.007353        0.525877  0.283090   1.857633   
ENSG00000288660.1      0.627493        1.753328  0.492552   3.559680   
ENSG00000288669.1      0.233161       -0.984425  0.676723  -1.454695   
ENSG00000288674.1      8.092787       -0.380347  0.129235  -2.943052   
ENSG00000288675.1     28.850430        1.534663  0.145

... done in 2.53 seconds.



### Extract significant genes

Keep genes with `padj < 0.05` and `|log2FoldChange| > 2`  
Save the gene ID list for use as the DESeq2 feature set in the ML model.

In [ ]:
res = ds.results_df
res_sig_pos = res.query("padj < 0.05 & 2 < log2FoldChange").sort_values("padj")
res_sig_neg = res.query("padj < 0.05 & log2FoldChange < -2").sort_values("padj")
res_sig = pd.concat(
    [res_sig_pos, 
    res_sig_neg], 
    axis=0
)
res_sig.sort_values("padj")

pd.DataFrame(res_sig.index).to_csv(os.path.join(os.getenv("Output_path"),"res_sig_list.txt"), header=False, index=False)

### Export the final tumor-only count matrix

Restrict the filtered count matrix to tumor samples only. 
The Control samples were only used for the Deseq2 analysis to define genes of interest.

In [14]:
tumor_ids = combined.loc[combined['sample_group'] == 'Tumeric_group', 'file_id'].unique()

countData = count_pv_sel.loc[count_pv_sel.index.intersection(tumor_ids)]

countData.index.name = 'file_id'
countData.to_csv(os.path.join(os.getenv("Output_path"),'Countdata.csv'))